In [ ]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
import os


In [ ]:
load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')
model_name = os.getenv('GEMINI_MODEL', 'gemini-2.5-flash')

model = ChatGoogleGenerativeAI(
    model=model_name,
    api_key=api_key,
)


In [ ]:
from langgraph.graph import add_messages


class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


In [ ]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = model.invoke(messages)
    return {'messages': [response]}


In [ ]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)


In [ ]:
# interactive chat loop with a persisted thread
thread_id = '1'

while True:
    user_message = input('Type here: ')

    if not user_message.strip():
        continue
    print('User: ', user_message)
    if user_message.strip().lower() in ('exit', 'quit', 'bye'):
        break

    config = {'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke(
        {'messages': [HumanMessage(content=user_message)]},
        config=config,
    )

    print('AI: ', response['messages'][-1].content)
